In [1]:
import stanza
import pandas as pd
# stanza.download("pt")
nlp = stanza.Pipeline("pt", download_method=None)

2026-06-03 10:30:55 INFO: Loading these models for language: pt (Portuguese):
| Processor    | Package         |
----------------------------------
| tokenize     | bosque          |
| mwt          | bosque          |
| pos          | bosque_charlm   |
| lemma        | bosque_nocharlm |
| constituency | cintil_charlm   |
| depparse     | bosque_charlm   |

2026-06-03 10:30:55 INFO: Using device: cuda
2026-06-03 10:30:55 INFO: Loading: tokenize
2026-06-03 10:30:56 INFO: Loading: mwt
2026-06-03 10:30:56 INFO: Loading: pos
2026-06-03 10:30:59 INFO: Loading: lemma
2026-06-03 10:30:59 INFO: Loading: constituency
2026-06-03 10:31:00 INFO: Loading: depparse
2026-06-03 10:31:00 INFO: Done loading processors!


In [2]:
VERB_COLUMNS = ["text", "lemma", "VerbForm", "Mood", "Tense", "Person", "Number", "Gender", "Voice"]
verbs_df = pd.DataFrame(columns=VERB_COLUMNS)

def process_verb(verb):
    global verbs_df
    
    new_row = {
        "text": verb.text,
        "lemma": verb.lemma
    }

    feats_dict = dict(feat.split("=") for feat in verb.feats.split("|"))

    new_row = new_row | feats_dict

    df_row = pd.DataFrame([new_row])
    verbs_df = pd.concat([verbs_df, df_row], ignore_index=True)

In [3]:
CONTRACTION_COLUMNS = ["text"]
contraction_df = pd.DataFrame(columns=CONTRACTION_COLUMNS)

def process_contraction(contraction):
    global contraction_df

    new_row = {
        "text": contraction.text
    }

    for i, word in enumerate(contraction.words):
        new_row[f"text{i+1}"] = word.text
        new_row[f"lemma{i+1}"] = word.lemma
    
    df_row = pd.DataFrame([new_row])
    contraction_df = pd.concat([contraction_df, df_row], ignore_index=True)

In [6]:
def extract_text_info(text):
    global contraction_df
    global verbs_df

    doc = nlp(text)
    for sent in doc.sentences:
        for token in sent.tokens:
            # Check if word is a contraction
            if len(token.words) > 1:
                process_contraction(token)
            else:
                word = token.words[0]
                if word.pos == "VERB":
                    process_verb(word)

    contraction_df = contraction_df.drop_duplicates()
    verbs_df = verbs_df.drop_duplicates()

In [7]:
text = """
Algum tempo hesitei se devia abrir estas memorias pelo principio ou
pelo fim, isto é, se poria em primeiro logar o meu nascimento ou a
minha morte. Supposto o uso vulgar seja começar pelo nascimento, duas
considerações me levaram a adoptar differente methodo: a primeira é que
eu não sou propriamente um autor defunto, mas um defunto autor, para
quem a campa foi outro berço; a segunda é que o escripto ficaria assim
mais galante e mais novo. Moysés, que tambem contou a sua morte, não a
poz no introito, mas no cabo: differença radical entre este livro e o
Pentateuco.

Dito isto, expirei ás duas horas da tarde de uma sexta feira do mez de
agosto de 1869, na minha bella chacara de Catumby. Tinha uns sessenta
e quatro annos, rijos e prosperos, era solteiro, possuia cerca de
tresentos contos e fui acompanhado ao cemiterio por onze amigos.
Onze amigos! Verdade é que não houve cartas nem annuncios. Accresce
que chovia--peneirava--uma chuvinha miuda, triste e constante, tão
constante e tão triste, que levou um daquelles fieis da ultima hora
a intercalar esta engenhosa idéa no discurso que proferiu á beira de
minha cova:--«Vós, que o conhecestes, meus senhores, vós podeis dizer
commigo que a natureza parece estar chorando a perda irreparavel de
um dos mais bellos caracteres que tem honrado a humanidade. Este ar
sombrio, estas gotas do ceu, aquellas nuvens escuras que cobrem o
azul como um crepe funereo, tudo isso é a dor crua e má que lhe róe á
natureza as mais intimas entranhas; tudo isso é um sublime louvor ao
nosso illustre finado.»

Bom e fiel amigo! Não, não me arrependo das vinte apolices que lhe
deixei. E foi assim que cheguei á clausula dos meus dias; foi assim que
me encaminhei para o _undiscovered country_ de Hamlet, sem as ancias
nem as duvidas do moço principe, mas pausado e tropego, como quem se
retira tarde do expectaculo. Tarde e aborrecido. Viram-me ir umas nove
ou dez pessoas, entre ellas tres senhoras,--minha irmã Sabina, casada
com o Cotrim,--a filha, um lyrio do valle,--e... Tenham paciência!
daqui a pouco lhes direi quem era a terceira senhora. Contentem-se de
saber que essa anonyma, ainda que não parenta, padeceu mais do que as
parentas. É verdade, padeceu mais. Não digo que se carpisse, não digo
que se deixasse rolar pelo chão, epileptica. Nem o meu óbito era cousa
altamente dramatica... Um solteirão que expira aos sessenta e quatro
annos, não parece que reuna em si todos os elementos de uma tragedia. E
dado que sim, o que menos convinha a essa anonyma era apparental-o. De
pé, á cabeceira da cama, com os olhos estúpidos, a boca entreaberta, a
triste senhora mal podia crêr na minha extincção.

--Morto! morto! dizia comsigo.

E a imaginação della, como as cegonhas que um illustre viajante viu
desferirem o vôo desde o Illysso ás ribas africanas, sem embargo das
ruinas e dos tempos,--a imaginação dessa senhora tambem voou por
sobre os destroços presentes até ás ribas de uma Africa juvenil...
Deixal-a ir; lá iremos mais tarde; lá iremos quando eu me restituir aos
primeiros annos. Agora, quero morrer tranquillamente, methodicamente,
ouvindo os soluços das damas, as fallas baixas dos homens, a chuva que
tamborila nas folhas de tinhorão da chacara, e o som estridulo de uma
navalha que um amolador está afiando lá fóra, á porta de um correeiro.
Juro-lhes que essa orchestra da morte foi muito menos triste do que
podia parecer; e de certo ponto em deante chegou a ser deliciosa.
A vida estrebuchava-me no peito, com uns impetos de vaga marinha,
esvaia-se-me a consciencia, eu descia á immobilidade physica e moral,
e o corpo fazia-se-me planta, e pedra, e lodo, e cousa nenhuma.

Morri de uma pneumonia; mas se lhe disser que foi menos a pneumonia, do
que uma idéa grandiosa e util, a causa da minha morte, é possivel que o
leitor me não creia, e todavia é verdade. Vou expor-lhe summariamente o
caso. Julgue-o por si mesmo.
"""

text = text.strip()

extract_text_info(text)

In [8]:
verbs_df

,text,lemma,VerbForm,Mood,Tense,Person,Number,Gender,Voice,ExtPos
0,hesitei,hesitar,Fin,Ind,Past,1,Sing,NaN,NaN,NaN
1,devia,dever,Fin,Ind,Imp,3,Sing,NaN,NaN,NaN
2,abrir,abrir,Inf,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,é,ser,Fin,Ind,Pres,3,Sing,NaN,NaN,NaN
4,poria,poriar,Fin,Cnd,NaN,3,Sing,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
77,chegou,chegar,Fin,Ind,Past,3,Sing,NaN,NaN,AUX
78,descia,descer,Fin,Ind,Imp,1,Sing,NaN,NaN,NaN
79,Morri,morrer,Fin,Ind,Past,1,Sing,NaN,NaN,NaN
80,disser,dizer,Fin,Sub,Fut,3,Sing,NaN,NaN,NaN


In [9]:
contraction_df

,text,text1,lemma1,text2,lemma2
0,pelo,por,por,o,o
3,no,em,em,o,o
5,da,de,de,a,o
6,do,de,de,o,o
7,na,em,em,a,o
8,ao,a,a,o,o
9,daquelles,de,de,aqueles,aquele
12,dos,de,de,os,o
15,das,de,de,as,o
19,Viram-me,Viram,ver,me,eu
